In [8]:
from dotenv import load_dotenv
load_dotenv()  # 必须要这句！
import warnings
warnings.filterwarnings("ignore", message=".*extra_body.*")

In [9]:
import os
import uuid
import requests
from typing import Annotated
from typing_extensions import TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.base import BaseStore
from langgraph.store.memory import InMemoryStore
from langchain_core.embeddings import Embeddings

# --- 1. 核心修复：纯原生百炼嵌入类 (规避所有 SDK 导致的 400 错误) ---
class PureDashScopeEmbeddings(Embeddings):
    def __init__(self, model: str, api_key: str):
        self.model = model
        self.api_key = api_key
        self.url = "https://dashscope.aliyuncs.com/compatible-mode/v1/embeddings"

    def embed_query(self, text: str) -> list[float]:
        return self._embed([text])[0] #把text文本转换成向量

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self._embed(texts) #把多句话转成向量

    def _embed(self, texts: list[str]) -> list[list[float]]: #真正发请求给阿里云，拿到向量
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        # 极简 Payload，只传百炼认得的字段
        payload = {"model": self.model, "input": texts}
        response = requests.post(self.url, headers=headers, json=payload) #json 把payload转换成json格式让python看得懂
        if response.status_code != 200:
            raise Exception(f"百炼嵌入 API 报错: {response.text}")
        return [item["embedding"] for item in response.json()["data"]]

# --- 2. 初始化组件 ---
api_key = os.environ.get("DASHSCOPE_API_KEY")
base_url = "https://dashscope.aliyuncs.com/compatible-mode/v1"

# 嵌入模型
embeddings = PureDashScopeEmbeddings(model="text-embedding-v3", api_key=api_key)

# 聊天模型 (qwen-plus)
model = ChatOpenAI(
    model="qwen-plus",
    temperature=0,
    api_key=api_key,
    base_url=base_url
)

# 长期存储 (Store)
in_memory_store = InMemoryStore(
    index={"embed": embeddings, "dims": 1024}
)

# --- 3. 定义节点逻辑 ---
def call_model(state: MessagesState, config: RunnableConfig, *, store: BaseStore):
    # 获取 ID 和 命名空间
    user_id = config["configurable"].get("user_id", "default_user")
    namespace = ("memories", user_id)

    # 提取纯文本输入
    last_msg = state["messages"][-1]
    query_text = str(last_msg.content).strip()

    # A. 检索记忆
    info = ""
    if query_text:
        try:
            memories = store.search(namespace, query=query_text, limit=2)
            if memories:
                info = "\n".join([d.value["data"] for d in memories if "data" in d.value])
                print(f"--- [读取记忆成功]: {info} ---")
        except Exception as e:
            print(f"--- [检索失败调试]: {e} ---")

    # B. 存储记忆 (如果包含关键词)
    if "记住" in query_text:
        # 简单清洗，提取名字
        fact = query_text.replace("请记住", "").replace("记住", "").replace("我叫", "").strip()
        if fact:
            memory_item = {"data": f"用户名字是 {fact}"}
            store.put(namespace, str(uuid.uuid4()), memory_item)
            print(f"--- [写入记忆成功]: {memory_item} ---")

    # C. 构造消息并调用模型
    system_msg = f"你是一个有记忆的小助手。已知用户信息：\n{info if info else '暂无记录'}"
    prompt = [{"role": "system", "content": system_msg}] + state["messages"]

    response = model.invoke(prompt)
    return {"messages": response}

# --- 4. 构建并编译图 ---
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

# 绑定短期记忆 (MemorySaver) 和 长期存储 (Store)
graph = builder.compile(checkpointer=MemorySaver(), store=in_memory_store)

# --- 5. 模拟问问题 ---
config = {"configurable": {"thread_id": "thread_abc", "user_id": "user_tomie"}}

print("\n==== 第一轮对话：存入记忆 ====")
input_1 = {"role": "user", "content": "请记住我的名字叫tomiezhang"}
for chunk in graph.stream({"messages": [input_1]}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

print("\n==== 第二轮对话：检索记忆 ====")
# 稍微换个问法，测试语义搜索 (search) 的威力
input_2 = {"role": "user", "content": "嘿，你还记得我叫什么吗？"}
for chunk in graph.stream({"messages": [input_2]}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()


==== 第一轮对话：存入记忆 ====
================================ Human Message =================================

请记住我的名字叫tomiezhang
--- [写入记忆成功]: {'data': '用户名字是 我的名字叫tomiezhang'} ---
================================== Ai Message ==================================

好的，我已经记住了你的名字是 **tomiezhang**！😊  
很高兴认识你～ 如果有任何问题、需要帮助，或者只是想聊聊天，随时欢迎！

==== 第二轮对话：检索记忆 ====
================================ Human Message =================================

嘿，你还记得我叫什么吗？
--- [读取记忆成功]: 用户名字是 我的名字叫tomiezhang ---
================================== Ai Message ==================================

当然记得！你叫 **tomiezhang** 😊  
（名字简洁有力，还带点技术感～）  
有什么想聊的、需要帮忙的，随时告诉我！
